
> **Public reproducibility notebook.** This notebook contains the experiment logic used in the paper. Machine-specific paths have been replaced with repository-relative configuration, and saved outputs are intentionally cleared.


# Candidate-Pool Coverage Analysis
## Is Pattern Top-100 candidate generation the bottleneck?

This is the **final planned experiment**. It requires **no model training**.

The preceding similarity-robustness experiment showed that a strong
**last-value-anchored / offset-normalized L2** retrieval rule can outperform
the learned reranker in some domains. However, the comparison is asymmetric:

- **Full L2** retrieves directly from the entire temporally admissible same-channel memory.
- **Ours** can only rerank candidates already included in **Pattern Top-100**.

We therefore measure

\[
\mathrm{Coverage@10}
=
\frac{
|\mathrm{Top10}_{\mathrm{L2,full}}
\cap
\mathrm{Top100}_{\mathrm{Pattern}}|
}{10},
\]

and also evaluate **L2-within-Pattern-100**, which applies exactly the same L2
rule but is restricted to the candidate pool available to Ours.

This separates:

- **Full L2 vs L2-within-Pattern-100:** candidate-generation penalty.
- **Ours vs Full L2:** end-to-end comparison.
- **Ours vs L2-within-Pattern-100:** reranking comparison with the pool fixed.

Frozen setup: Electricity, Traffic, Exchange, Solar; \(L=96\);
\(H\in\{24,48,96\}\); same-channel retrieval; \(M=100\); \(K=10\);
moving-block bootstrap with 5,000 replicates.

> **Public repository version.** Paths are repository-relative by default.
> Set `WHM_DATA_ROOT` to use datasets stored elsewhere and `WHM_WORK_ROOT` to move generated caches/checkpoints outside the repository.
> Saved execution outputs were cleared intentionally so the notebook does not expose machine-specific paths or stale results.


In [ ]:
from pathlib import Path
import os

def _find_repo_root(start=None):
    """Locate the repository root from the current working directory."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "experiments").exists():
            return candidate
    raise RuntimeError(
        "Repository root not found. Start Jupyter from inside the cloned "
        "which-histories-matter repository, or set WHM_REPO_ROOT."
    )

_env_repo = os.environ.get("WHM_REPO_ROOT")
REPO_ROOT = Path(_env_repo).expanduser().resolve() if _env_repo else _find_repo_root()
REPO_DATA_ROOT = Path(os.environ.get("WHM_DATA_ROOT", REPO_ROOT / "data")).expanduser().resolve()
REPO_WORK_ROOT = Path(os.environ.get("WHM_WORK_ROOT", REPO_ROOT / "_work")).expanduser().resolve()
REPO_WORK_ROOT.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Data root:", REPO_DATA_ROOT)
print("Work root:", REPO_WORK_ROOT)


## 0. Environment and frozen paths

In [ ]:

from pathlib import Path
import math
import warnings
import random

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

warnings.filterwarnings("ignore")

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

DATA_ROOT = REPO_DATA_ROOT

DATA_PATHS = {
    "Electricity":
        DATA_ROOT / "electricity/electricity.csv",

    "Traffic":
        DATA_ROOT / "traffic/traffic.csv",

    "Exchange":
        DATA_ROOT / "exchange_rate/exchange_rate.csv",

    "Solar":
        DATA_ROOT / "Solar/solar_AL.txt",
}

RESULT_DIR = REPO_WORK_ROOT / "final_confirmatory"

CACHE_DIR = RESULT_DIR / "cache"

SARAF_DIR = (
    RESULT_DIR /
    "external_baselines" /
    "saraf_matched"
)

OUT_DIR = (
    RESULT_DIR /
    "similarity_robustness"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

DATASETS = [
    "Electricity",
    "Traffic",
    "Exchange",
    "Solar",
]

HORIZONS = [24, 48, 96]

SEQ_LEN = 96
TOP_M = 100
TOP_K = 10

MAX_MEMORY_WINDOWS = 50000
BLOCK_ANCHORS = 10
N_BOOT = 5000
EPS = 1e-8

# Exact confirmatory settings.
MAX_CHANNELS = {
    "Electricity": 32,
    "Traffic": 32,
    "Exchange": None,
    "Solar": None,
}

DATASET_SEED = {
    "Electricity": 3303,
    "Traffic": 4404,
    "Exchange": 5505,
    "Solar": 6606,
}

QUERY_BATCH = 256

print("Device:", DEVICE)
print("Confirmatory directory:", RESULT_DIR)
print("Output directory:", OUT_DIR)

assert RESULT_DIR.exists()
assert CACHE_DIR.exists()

for name, path in DATA_PATHS.items():
    assert path.exists(), (name, path)


## 1. Verify the frozen confirmatory cache

In [ ]:

required = [
    RESULT_DIR / "00_data_manifest.csv",
]

for dataset_name in DATASETS:
    for H in HORIZONS:
        required.extend([
            CACHE_DIR / f"{dataset_name}_H{H}_windows.npz",
            CACHE_DIR / f"{dataset_name}_H{H}_meta.csv.gz",
            CACHE_DIR / f"{dataset_name}_H{H}_same_topM.npz",
            RESULT_DIR / f"query_level_{dataset_name}_H{H}.csv.gz",
        ])

missing = [str(p) for p in required if not p.exists()]

if missing:
    print("Missing files:")
    for p in missing:
        print(" -", p)
    raise FileNotFoundError(
        "Run the final confirmatory experiment first."
    )

print("All frozen confirmatory files are available.")


## 2. Exact temporal split and balanced memory sampling

In [ ]:

manifest = pd.read_csv(
    RESULT_DIR / "00_data_manifest.csv"
)

N_ROWS = {
    row["Dataset"]: int(row["Rows"])
    for _, row in manifest.iterrows()
    if row["Dataset"] in DATASETS
}

assert set(N_ROWS) == set(DATASETS)


def split_boundaries(n):
    train_end = int(0.70 * n)
    val_end = int(0.80 * n)

    return {
        "train_end": train_end,
        "val_end": val_end,
    }


SPLITS = {
    d: split_boundaries(N_ROWS[d])
    for d in DATASETS
}


def balanced_subset(
    meta,
    indices,
    max_n,
    seed,
):
    indices = np.asarray(
        indices,
        dtype=np.int64,
    )

    if len(indices) <= max_n:
        return np.sort(indices)

    rng = np.random.default_rng(seed)

    channels = (
        meta.iloc[indices]["ChannelIndex"]
        .to_numpy(dtype=np.int64)
    )

    unique_channels = np.unique(channels)
    random_channel_order = rng.permutation(
        unique_channels
    )

    base = max_n // len(unique_channels)
    extra = max_n % len(unique_channels)

    chosen_parts = []

    for rank, c in enumerate(random_channel_order):
        pos = indices[channels == c]
        quota = base + (
            1 if rank < extra else 0
        )
        take = min(quota, len(pos))

        if take > 0:
            chosen_parts.append(
                rng.choice(
                    pos,
                    size=take,
                    replace=False,
                )
            )

    chosen = np.unique(
        np.concatenate(chosen_parts)
    )

    if len(chosen) < max_n:
        remaining = np.setdiff1d(
            indices,
            chosen,
            assume_unique=False,
        )

        add_n = min(
            max_n - len(chosen),
            len(remaining),
        )

        if add_n > 0:
            chosen = np.concatenate([
                chosen,
                rng.choice(
                    remaining,
                    size=add_n,
                    replace=False,
                ),
            ])

    return np.sort(
        chosen.astype(np.int64)
    )


## 3. Recover the exact frozen test memory

In [ ]:

def infer_test_memory(
    dataset_name,
    H,
    meta,
    topm,
):
    """
    Reconstruct the exact frozen test-memory sample used by the
    final confirmatory notebook.

    Confirmatory rule:
        base = DATASET_SEED[dataset] + H * 10
        test_memory seed = base + 5
    """
    val_end = SPLITS[
        dataset_name
    ]["val_end"]

    future_end = (
        meta["FutureEnd"]
        .to_numpy(dtype=np.int64)
    )

    full_test_memory = np.where(
        future_end < val_end
    )[0].astype(np.int64)

    base = (
        DATASET_SEED[dataset_name] +
        H * 10
    )

    memory = balanced_subset(
        meta,
        full_test_memory,
        MAX_MEMORY_WINDOWS,
        base + 5,
    )

    cached_candidates = np.unique(
        topm["test_idx"]
        .astype(np.int64)
        .reshape(-1)
    )

    coverage = float(
        np.isin(
            cached_candidates,
            memory,
        ).mean()
    )

    if coverage < 0.999999:
        raise RuntimeError(
            f"Exact test-memory reconstruction failed for "
            f"{dataset_name}, H={H}. "
            f"Cached-candidate coverage={coverage:.8f}."
        )

    return memory, {
        "Mode": "exact_confirmatory_seed",
        "SeedOffset": 5,
        "CachedCandidateCoverage": coverage,
    }


## 4. Reconstruct the raw past windows

In [ ]:

def load_standard_csv(path):
    df = pd.read_csv(path)

    timestamp_cols = [
        c for c in df.columns
        if str(c).lower() in {
            "date",
            "datetime",
            "timestamp",
            "time",
        }
    ]

    x = (
        df
        .drop(
            columns=timestamp_cols,
            errors="ignore",
        )
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
    )

    good_cols = [
        c for c in x.columns
        if x[c].notna().mean() > 0.99
    ]

    x = x[good_cols]

    x = (
        x
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .interpolate(
            axis=0,
            limit_direction="both",
        )
        .ffill()
        .bfill()
    )

    assert x.shape[1] > 0
    assert np.isfinite(
        x.to_numpy(dtype=np.float32)
    ).all()

    return x


def load_solar_txt(path):
    x = pd.read_csv(
        path,
        header=None,
    )

    if x.shape[1] == 1:
        x = pd.read_csv(
            path,
            header=None,
            sep=r"\s+",
        )

    x = (
        x
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .interpolate(
            axis=0,
            limit_direction="both",
        )
        .ffill()
        .bfill()
    )

    assert x.shape[1] == 137

    x.columns = [
        f"Solar_{i:03d}"
        for i in range(x.shape[1])
    ]

    return x


RAW = {
    "Electricity":
        load_standard_csv(
            DATA_PATHS["Electricity"]
        ),

    "Traffic":
        load_standard_csv(
            DATA_PATHS["Traffic"]
        ),

    "Exchange":
        load_standard_csv(
            DATA_PATHS["Exchange"]
        ),

    "Solar":
        load_solar_txt(
            DATA_PATHS["Solar"]
        ),
}


SELECTED_CHANNELS = {}
CHANNEL_NORMALIZED = {}

for dataset_name in DATASETS:
    df = RAW[dataset_name]

    train_end = SPLITS[
        dataset_name
    ]["train_end"]

    train = df.iloc[:train_end]

    train_std = train.std(
        axis=0,
        ddof=0,
    )

    valid_cols = [
        c for c in df.columns
        if (
            np.isfinite(train_std[c])
            and train_std[c] > 1e-6
        )
    ]

    max_c = MAX_CHANNELS[
        dataset_name
    ]

    if (
        max_c is not None
        and len(valid_cols) > max_c
    ):
        idx = np.linspace(
            0,
            len(valid_cols) - 1,
            max_c,
            dtype=int,
        )

        selected = [
            valid_cols[i]
            for i in idx
        ]
    else:
        selected = valid_cols

    SELECTED_CHANNELS[
        dataset_name
    ] = selected

    df_sel = df[selected]

    mu = (
        df_sel.iloc[:train_end]
        .mean(axis=0)
        .to_numpy(dtype=np.float32)
    )

    sd = (
        df_sel.iloc[:train_end]
        .std(axis=0, ddof=0)
        .to_numpy(dtype=np.float32)
    )

    assert np.all(sd > 1e-6)

    arr = df_sel.to_numpy(
        dtype=np.float32
    )

    z = (
        arr -
        mu[None, :]
    ) / sd[None, :]

    assert np.isfinite(z).all()

    CHANNEL_NORMALIZED[
        dataset_name
    ] = z.astype(np.float32)

    print(
        dataset_name,
        "| reconstructed channels:",
        len(selected),
        "| rows:",
        len(z),
    )


def extract_past_windows(
    dataset_name,
    meta,
    indices,
):
    """
    Reconstruct [N, SEQ_LEN] train-normalized past windows exactly
    from cached metadata.
    """
    indices = np.asarray(
        indices,
        dtype=np.int64,
    )

    arr = CHANNEL_NORMALIZED[
        dataset_name
    ]

    sub = meta.iloc[
        indices
    ]

    channels = sub[
        "ChannelIndex"
    ].to_numpy(
        dtype=np.int64
    )

    anchors = sub[
        "Anchor"
    ].to_numpy(
        dtype=np.int64
    )

    out = np.empty(
        (
            len(indices),
            SEQ_LEN,
        ),
        dtype=np.float32,
    )

    for j, (
        c,
        anchor,
    ) in enumerate(
        zip(
            channels,
            anchors,
        )
    ):
        out[j] = arr[
            anchor - SEQ_LEN:
            anchor,
            c,
        ]

    return out


TASK = {}
memory_rows = []

for dataset_name in DATASETS:
    for H in HORIZONS:
        key = (
            dataset_name,
            H,
        )

        w = np.load(
            CACHE_DIR /
            f"{dataset_name}_H{H}_windows.npz"
        )

        meta = pd.read_csv(
            CACHE_DIR /
            f"{dataset_name}_H{H}_meta.csv.gz"
        )

        topm = np.load(
            CACHE_DIR /
            f"{dataset_name}_H{H}_same_topM.npz"
        )

        # Cache is intentionally compact.
        assert set(
            [
                "pattern",
                "context",
                "future",
            ]
        ).issubset(
            set(w.files)
        )

        q_idx = np.asarray(
            topm["test_query"],
            dtype=np.int64,
        )

        cached_topm_idx = np.asarray(
            topm["test_idx"],
            dtype=np.int64,
        )

        cached_topm_score = np.asarray(
            topm["test_score"],
            dtype=np.float32,
        )

        memory_idx, memory_info = infer_test_memory(
            dataset_name,
            H,
            meta,
            topm,
        )

        # Reconstruct only the frozen query and memory windows, not all
        # cached windows. This keeps memory use modest.
        q_past = extract_past_windows(
            dataset_name,
            meta,
            q_idx,
        )

        mem_past = extract_past_windows(
            dataset_name,
            meta,
            memory_idx,
        )

        future_all = np.asarray(
            w["future"],
            dtype=np.float32,
        )

        q_future = future_all[
            q_idx
        ].copy()

        mem_future = future_all[
            memory_idx
        ].copy()

        q_channel = (
            meta.iloc[q_idx]["ChannelIndex"]
            .to_numpy(dtype=np.int64)
        )

        q_anchor = (
            meta.iloc[q_idx]["Anchor"]
            .to_numpy(dtype=np.int64)
        )

        mem_channel = (
            meta.iloc[memory_idx]["ChannelIndex"]
            .to_numpy(dtype=np.int64)
        )

        # Map global cached-window index -> row in mem_past/mem_future.
        global_to_memory = np.full(
            len(meta),
            -1,
            dtype=np.int64,
        )

        global_to_memory[
            memory_idx
        ] = np.arange(
            len(memory_idx),
            dtype=np.int64,
        )

        TASK[key] = {
            "q_idx": q_idx,
            "q_past": q_past,
            "q_future": q_future,
            "q_channel": q_channel,
            "q_anchor": q_anchor,

            "memory_idx": memory_idx,
            "memory_past": mem_past,
            "memory_future": mem_future,
            "memory_channel": mem_channel,
            "global_to_memory": global_to_memory,

            "cached_topm_idx": cached_topm_idx,
            "cached_topm_score": cached_topm_score,
        }

        memory_rows.append({
            "Dataset": dataset_name,
            "Horizon": H,
            "Queries": len(q_idx),
            "MemoryWindows": len(memory_idx),
            **memory_info,
        })

        # Explicitly release the full future cache before the next task.
        del future_all
        w.close()
        topm.close()

memory_table = pd.DataFrame(
    memory_rows
)

display(memory_table)

memory_table.to_csv(
    OUT_DIR /
    "00_memory_reconstruction.csv",
    index=False,
)


## 5. Offset-normalized L2 retrieval

In [ ]:

def offset_l2_feature(x):
    return (
        x -
        x[:, -1:]
    ).astype(np.float32)


def topk_negative_l2_gpu(
    q_feat,
    m_feat,
    k,
    batch_size=QUERY_BATCH,
):
    q_feat = np.asarray(
        q_feat,
        dtype=np.float32,
    )

    m_feat = np.asarray(
        m_feat,
        dtype=np.float32,
    )

    m = torch.from_numpy(
        m_feat
    ).to(DEVICE)

    m_sq = (
        m.pow(2)
        .sum(dim=1)
        [None, :]
    )

    L = float(
        m_feat.shape[1]
    )

    all_idx = []
    all_score = []

    for start in range(
        0,
        len(q_feat),
        batch_size,
    ):
        q = torch.from_numpy(
            q_feat[
                start:
                start + batch_size
            ]
        ).to(DEVICE)

        q_sq = (
            q.pow(2)
            .sum(dim=1)
            [:, None]
        )

        dist2 = (
            q_sq +
            m_sq -
            2.0 * (
                q @ m.T
            )
        )

        score = -dist2 / L

        val, idx = torch.topk(
            score,
            k=k,
            dim=1,
            largest=True,
            sorted=True,
        )

        all_idx.append(
            idx.cpu().numpy()
        )

        all_score.append(
            val.cpu().numpy()
        )

    return (
        np.concatenate(
            all_idx,
            axis=0,
        ),
        np.concatenate(
            all_score,
            axis=0,
        ),
    )


@torch.no_grad()
def retrieve_full_l2(task, k=TOP_K):
    q_past = task["q_past"]
    memory_past = task["memory_past"]
    q_channel = task["q_channel"]
    memory_idx = task["memory_idx"]
    memory_channel = task["memory_channel"]

    selected_global = np.empty(
        (len(q_past), k),
        dtype=np.int64,
    )

    selected_score = np.empty(
        (len(q_past), k),
        dtype=np.float32,
    )

    for c in np.unique(q_channel):
        q_pos = np.where(
            q_channel == c
        )[0]

        mem_pos = np.where(
            memory_channel == c
        )[0]

        q_feat = offset_l2_feature(
            q_past[q_pos]
        )

        m_feat = offset_l2_feature(
            memory_past[mem_pos]
        )

        local_idx, local_score = (
            topk_negative_l2_gpu(
                q_feat,
                m_feat,
                k,
            )
        )

        selected_global[q_pos] = (
            memory_idx[
                mem_pos[
                    local_idx
                ]
            ]
        )

        selected_score[q_pos] = (
            local_score
        )

    return selected_global, selected_score


## 6. Validate the reconstructed Pattern Top-100 pool

In [ ]:

def l2_normalize_np(x):
    denom = np.linalg.norm(
        x,
        axis=1,
        keepdims=True,
    )

    return (
        x /
        np.maximum(
            denom,
            EPS,
        )
    ).astype(np.float32)


def pearson_feature(x):
    centered = (
        x -
        x.mean(
            axis=1,
            keepdims=True,
        )
    )

    return l2_normalize_np(
        centered.astype(np.float32)
    )


def topk_cosine_gpu(
    q_feat,
    m_feat,
    k,
    batch_size=QUERY_BATCH,
):
    q_feat = np.asarray(
        q_feat,
        dtype=np.float32,
    )

    m_feat = np.asarray(
        m_feat,
        dtype=np.float32,
    )

    m = torch.from_numpy(
        m_feat
    ).to(DEVICE)

    all_idx = []

    for start in range(
        0,
        len(q_feat),
        batch_size,
    ):
        q = torch.from_numpy(
            q_feat[
                start:
                start + batch_size
            ]
        ).to(DEVICE)

        score = q @ m.T

        _, idx = torch.topk(
            score,
            k=k,
            dim=1,
            largest=True,
            sorted=True,
        )

        all_idx.append(
            idx.cpu().numpy()
        )

    return np.concatenate(
        all_idx,
        axis=0,
    )


@torch.no_grad()
def reconstruct_pattern_topm(task):
    q_past = task["q_past"]
    memory_past = task["memory_past"]
    q_channel = task["q_channel"]
    memory_idx = task["memory_idx"]
    memory_channel = task["memory_channel"]

    selected_global = np.empty(
        (len(q_past), TOP_M),
        dtype=np.int64,
    )

    for c in np.unique(q_channel):
        q_pos = np.where(
            q_channel == c
        )[0]

        mem_pos = np.where(
            memory_channel == c
        )[0]

        q_feat = pearson_feature(
            q_past[q_pos]
        )

        m_feat = pearson_feature(
            memory_past[mem_pos]
        )

        local_idx = topk_cosine_gpu(
            q_feat,
            m_feat,
            TOP_M,
        )

        selected_global[q_pos] = (
            memory_idx[
                mem_pos[
                    local_idx
                ]
            ]
        )

    return selected_global


validation_rows = []

for dataset_name in DATASETS:
    for H in HORIZONS:
        task = TASK[
            (dataset_name, H)
        ]

        reconstructed = (
            reconstruct_pattern_topm(
                task
            )
        )

        cached = task[
            "cached_topm_idx"
        ]

        recall = np.mean([
            len(
                np.intersect1d(
                    a,
                    b,
                )
            ) / TOP_M
            for a, b in zip(
                reconstructed,
                cached,
            )
        ])

        validation_rows.append({
            "Dataset": dataset_name,
            "Horizon": H,
            "PatternRecallAt100": recall,
        })

        print(
            dataset_name,
            H,
            "Pattern Recall@100:",
            round(float(recall), 6),
        )

        if recall < 0.995:
            raise RuntimeError(
                f"Pattern pool reconstruction mismatch: "
                f"{dataset_name}, H={H}, recall={recall:.6f}"
            )

validation = pd.DataFrame(
    validation_rows
)

display(validation)


## 7. Coverage@10 and L2-within-Pattern-100

In [ ]:

def future_metrics_from_global_indices(
    task,
    selected_global,
):
    local = task[
        "global_to_memory"
    ][
        selected_global
    ]

    if np.any(local < 0):
        raise RuntimeError(
            "A selected candidate is not in the frozen test memory."
        )

    selected_future = task[
        "memory_future"
    ][local]

    q_future = task[
        "q_future"
    ]

    analog = np.mean(
        (
            selected_future -
            q_future[:, None, :]
        ) ** 2,
        axis=2,
    ).mean(axis=1)

    forecast = np.mean(
        (
            selected_future.mean(axis=1) -
            q_future
        ) ** 2,
        axis=1,
    )

    return (
        analog.astype(np.float32),
        forecast.astype(np.float32),
    )


def l2_within_pattern_pool(
    task,
    k=TOP_K,
):
    q_past = task["q_past"]
    cached_pool = task[
        "cached_topm_idx"
    ]

    global_to_memory = task[
        "global_to_memory"
    ]

    local_pool = global_to_memory[
        cached_pool
    ]

    if np.any(local_pool < 0):
        raise RuntimeError(
            "Cached Pattern candidate is missing from test memory."
        )

    cand_past = task[
        "memory_past"
    ][local_pool]

    q_feat = offset_l2_feature(
        q_past
    )

    cand_feat = (
        cand_past -
        cand_past[:, :, -1:]
    ).astype(np.float32)

    dist2 = np.mean(
        (
            cand_feat -
            q_feat[:, None, :]
        ) ** 2,
        axis=2,
    )

    local_topk = np.argpartition(
        dist2,
        kth=k - 1,
        axis=1,
    )[:, :k]

    row = np.arange(
        len(dist2)
    )[:, None]

    order = np.argsort(
        dist2[
            row,
            local_topk,
        ],
        axis=1,
    )

    local_topk = local_topk[
        row,
        order,
    ]

    return cached_pool[
        row,
        local_topk,
    ]


QUERY_RESULTS = {}
task_rows = []

for dataset_name in DATASETS:
    for H in HORIZONS:
        task = TASK[
            (dataset_name, H)
        ]

        full_l2_idx, _ = (
            retrieve_full_l2(
                task,
                TOP_K,
            )
        )

        pool_l2_idx = (
            l2_within_pattern_pool(
                task,
                TOP_K,
            )
        )

        pattern_pool = task[
            "cached_topm_idx"
        ]

        overlap_count = np.asarray([
            len(
                np.intersect1d(
                    full_row,
                    pool_row,
                )
            )
            for full_row, pool_row in zip(
                full_l2_idx,
                pattern_pool,
            )
        ], dtype=np.int64)

        coverage = (
            overlap_count /
            float(TOP_K)
        )

        full_analog, full_forecast = (
            future_metrics_from_global_indices(
                task,
                full_l2_idx,
            )
        )

        pool_analog, pool_forecast = (
            future_metrics_from_global_indices(
                task,
                pool_l2_idx,
            )
        )

        existing = pd.read_csv(
            RESULT_DIR /
            f"query_level_{dataset_name}_H{H}.csv.gz"
        )

        q = pd.DataFrame({
            "Anchor":
                task["q_anchor"],

            "ChannelIndex":
                task["q_channel"],

            "L2Full_CoverageCountInPattern100":
                overlap_count,

            "L2Full_CoverageAt10":
                coverage,

            "L2Full_AnalogFutureMSE":
                full_analog,

            "L2Pattern100_AnalogFutureMSE":
                pool_analog,

            "Learned_AnalogFutureMSE":
                existing[
                    "Learned_AnalogFutureMSE"
                ].to_numpy(),

            "Pattern_AnalogFutureMSE":
                existing[
                    "Pattern_AnalogFutureMSE"
                ].to_numpy(),

            "L2Full_RetrievalForecastMSE":
                full_forecast,

            "L2Pattern100_RetrievalForecastMSE":
                pool_forecast,

            "Learned_RetrievalForecastMSE":
                existing[
                    "Learned_RetrievalForecastMSE"
                ].to_numpy(),
        })

        QUERY_RESULTS[
            (dataset_name, H)
        ] = q

        l2_full = float(
            q[
                "L2Full_AnalogFutureMSE"
            ].mean()
        )

        l2_pool = float(
            q[
                "L2Pattern100_AnalogFutureMSE"
            ].mean()
        )

        learned = float(
            q[
                "Learned_AnalogFutureMSE"
            ].mean()
        )

        pattern = float(
            q[
                "Pattern_AnalogFutureMSE"
            ].mean()
        )

        task_rows.append({
            "Dataset": dataset_name,
            "Horizon": H,

            "MeanCoverageAt10":
                float(
                    q[
                        "L2Full_CoverageAt10"
                    ].mean()
                ),

            "MedianCoverageAt10":
                float(
                    q[
                        "L2Full_CoverageAt10"
                    ].median()
                ),

            "FullCoverageQueryFraction":
                float(
                    (
                        q[
                            "L2Full_CoverageAt10"
                        ] == 1.0
                    ).mean()
                ),

            "AtLeastHalfCoverageQueryFraction":
                float(
                    (
                        q[
                            "L2Full_CoverageAt10"
                        ] >= 0.5
                    ).mean()
                ),

            "Pattern":
                pattern,

            "Learned":
                learned,

            "L2Full":
                l2_full,

            "L2WithinPattern100":
                l2_pool,

            "CandidatePoolPenalty_%":
                100.0 *
                (
                    l2_pool -
                    l2_full
                ) /
                l2_full,

            "Ours_vs_L2Full_%":
                100.0 *
                (
                    l2_full -
                    learned
                ) /
                l2_full,

            "Ours_vs_L2WithinPattern100_%":
                100.0 *
                (
                    l2_pool -
                    learned
                ) /
                l2_pool,
        })

        q.to_csv(
            OUT_DIR /
            f"coverage_query_level_{dataset_name}_H{H}.csv.gz",
            index=False,
            compression="gzip",
        )

task_table = pd.DataFrame(
    task_rows
)

display(task_table)

task_table.to_csv(
    OUT_DIR /
    "07_candidate_pool_coverage_task_table.csv",
    index=False,
)


## 8. Moving-block bootstrap

In [ ]:

def moving_block_bootstrap(
    x,
    block_len,
    n_boot,
    seed,
):
    x = np.asarray(
        x,
        dtype=np.float64,
    )

    n = len(x)
    assert n >= block_len

    rng = np.random.default_rng(
        seed
    )

    n_blocks = math.ceil(
        n / block_len
    )

    max_start = (
        n -
        block_len
    )

    means = np.empty(
        n_boot,
        dtype=np.float64,
    )

    for b in range(n_boot):
        parts = []

        for _ in range(n_blocks):
            start = int(
                rng.integers(
                    0,
                    max_start + 1,
                )
            )

            parts.append(
                x[
                    start:
                    start + block_len
                ]
            )

        sample = np.concatenate(
            parts
        )[:n]

        means[b] = sample.mean()

    low = float(
        np.quantile(
            means,
            0.025,
        )
    )

    high = float(
        np.quantile(
            means,
            0.975,
        )
    )

    return {
        "ObservedImprovement":
            float(x.mean()),

        "CI_2.5%":
            low,

        "CI_97.5%":
            high,

        "SignificantPositive":
            bool(low > 0),

        "SignificantNegative":
            bool(high < 0),
    }


comparisons = [
    (
        "L2Pattern100_AnalogFutureMSE",
        "L2Full_AnalogFutureMSE",
        "CandidatePoolPenalty",
    ),

    (
        "L2Full_AnalogFutureMSE",
        "Learned_AnalogFutureMSE",
        "OursVsL2Full",
    ),

    (
        "L2Pattern100_AnalogFutureMSE",
        "Learned_AnalogFutureMSE",
        "OursVsL2WithinPattern100",
    ),
]

bootstrap_rows = []

for dataset_name in DATASETS:
    for H in HORIZONS:
        q = QUERY_RESULTS[
            (dataset_name, H)
        ]

        for left, right, label in comparisons:
            diff = (
                q[left] -
                q[right]
            )

            tmp = pd.DataFrame({
                "Anchor":
                    q["Anchor"],

                "Diff":
                    diff,
            })

            anchor_diff = (
                tmp
                .groupby("Anchor")["Diff"]
                .mean()
                .sort_index()
                .to_numpy()
            )

            result = moving_block_bootstrap(
                anchor_diff,
                BLOCK_ANCHORS,
                N_BOOT,
                seed=(
                    DATASET_SEED[
                        dataset_name
                    ]
                    +
                    H * 100
                    +
                    len(
                        bootstrap_rows
                    )
                ),
            )

            result.update({
                "Dataset":
                    dataset_name,

                "Horizon":
                    H,

                "Comparison":
                    label,
            })

            bootstrap_rows.append(
                result
            )

bootstrap = pd.DataFrame(
    bootstrap_rows
)

display(bootstrap)

bootstrap.to_csv(
    OUT_DIR /
    "08_candidate_pool_coverage_bootstrap.csv",
    index=False,
)


## 9. Dataset-level summary

In [ ]:

dataset_rows = []

for dataset_name in DATASETS:
    x = task_table[
        task_table[
            "Dataset"
        ] == dataset_name
    ]

    b = bootstrap[
        bootstrap[
            "Dataset"
        ] == dataset_name
    ]

    penalty_sig = int(
        b[
            b[
                "Comparison"
            ] ==
            "CandidatePoolPenalty"
        ]["SignificantPositive"].sum()
    )

    ours_pool_sig = int(
        b[
            b[
                "Comparison"
            ] ==
            "OursVsL2WithinPattern100"
        ]["SignificantPositive"].sum()
    )

    dataset_rows.append({
        "Dataset":
            dataset_name,

        "MeanCoverageAt10":
            float(
                x[
                    "MeanCoverageAt10"
                ].mean()
            ),

        "MeanCandidatePoolPenalty_%":
            float(
                x[
                    "CandidatePoolPenalty_%"
                ].mean()
            ),

        "CandidatePoolPenalty_SigHorizons":
            penalty_sig,

        "Mean_Ours_vs_L2Full_%":
            float(
                x[
                    "Ours_vs_L2Full_%"
                ].mean()
            ),

        "Mean_Ours_vs_L2WithinPattern100_%":
            float(
                x[
                    "Ours_vs_L2WithinPattern100_%"
                ].mean()
            ),

        "OursBeatsL2WithinPool_SigHorizons":
            ours_pool_sig,
    })

dataset_summary = pd.DataFrame(
    dataset_rows
)

display(dataset_summary)

dataset_summary.to_csv(
    OUT_DIR /
    "09_candidate_pool_coverage_dataset_summary.csv",
    index=False,
)

print("\nInterpretation:")
print(
    "- Low Coverage@10 + positive pool penalty: "
    "Pattern candidate generation is a bottleneck."
)
print(
    "- Ours loses to Full L2 but beats L2-within-Pattern-100: "
    "the Full-L2 advantage is largely a candidate-generation effect."
)
print(
    "- Ours also loses to L2-within-Pattern-100: "
    "the limitation is not only candidate generation."
)



# Interpretation guide

This analysis should be reported as a **diagnostic/limitation**, not as a new method.

### If candidate generation explains the Full-L2 advantage

If Coverage@10 is low, restricting L2 to Pattern Top-100 significantly degrades
L2, and Ours becomes competitive with or better than L2-within-Pattern-100:

> The Pattern candidate generator can exclude candidates preferred by a stronger
> alternative similarity. Once the candidate pool is held fixed, much of the
> Full-L2 advantage disappears, indicating that candidate generation - rather
> than future-supervised reranking alone - is an important remaining bottleneck.

### If L2 remains stronger inside the same pool

> Candidate generation explains only part of the gap. In these domains,
> last-value-anchored L2 remains a strong inductive bias even when evaluated
> inside the same Pattern candidate pool, highlighting a limitation of the
> current relevance objective.

Either outcome is scientifically useful.

**Stopping rule:** after this diagnostic, do not tune or add another retriever.
Move to manuscript revision.
